# EX_02 — Embeddings con Transformers (ejercicios)

**Notebook de referencia:** `notebook/02_Embeddings_Transformers.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Mean pooling

Con `AutoTokenizer` + `AutoModel`, obtén **last_hidden_state** para una frase y calcula el embedding de frase como media sobre tokens (excluyendo padding).


In [1]:
from transformers import AutoTokenizer, AutoModel
import torch

# 1. Cargamos un modelo ligero y su tokenizador
modelo_id = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(modelo_id)
model = AutoModel.from_pretrained(modelo_id)

text = "Transformers build contextual embeddings."

# 2. Tokenize: Convertimos texto a tensores de PyTorch
# Esto nos devuelve los 'input_ids' y la 'attention_mask' (para saber qué es padding)
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)

# 3. Forward pass: Pasamos los datos por el modelo
with torch.no_grad(): # Desactivamos gradientes porque no estamos entrenando
    outputs = model(**inputs)

# 4. Obtenemos el last_hidden_state (un vector por cada token de la frase)
token_embeddings = outputs.last_hidden_state

# 5. Mean pooling (ignorando el padding)
attention_mask = inputs['attention_mask']

# A) Expandimos la máscara de (batch, seq_len) a (batch, seq_len, hidden_size)
mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

# B) Multiplicamos los embeddings por la máscara (los padding se vuelven 0) y sumamos
sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)

# C) Dividimos por la cantidad de tokens REALES (evitando dividir por 0 con clamp)
num_tokens_reales = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)

sentence_embedding = sum_embeddings / num_tokens_reales

print(f"Texto original: '{text}'")
print(f"Forma de los embeddings de tokens: {token_embeddings.shape} -> (Batch, Tokens, Dimensiones)")
print(f"Forma del embedding de la frase (Mean Pool): {sentence_embedding.shape} -> (Batch, Dimensiones)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Texto original: 'Transformers build contextual embeddings.'
Forma de los embeddings de tokens: torch.Size([1, 11, 384]) -> (Batch, Tokens, Dimensiones)
Forma del embedding de la frase (Mean Pool): torch.Size([1, 384]) -> (Batch, Dimensiones)


## Actividad 2 — `sentence-transformers`

Usa `SentenceTransformer` para embedder dos frases y calcula similitud coseno. Comenta brevemente (en inglés en un comentario) por qué suele ser mejor que mean-pooling manual de BERT base.


In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np

# 1. Cargamos el modelo SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Definimos y codificamos (encode) dos frases semánticamente similares
frases = [
    "El gato duerme tranquilamente en el sofá.",
    "Un felino descansa plácidamente sobre el sillón."
]
embeddings = model.encode(frases)

# 3. Calculamos la similitud coseno manualmente con NumPy
v1 = embeddings[0]
v2 = embeddings[1]
similitud = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

print(f"Frase 1: '{frases[0]}'")
print(f"Frase 2: '{frases[1]}'")
print(f"Similitud Coseno: {similitud:.4f}")

# =====================================================================
# WHY SENTENCE-TRANSFORMERS IS BETTER THAN MANUAL MEAN-POOLING (BERT BASE)
# =====================================================================
# Base BERT models are trained for token-level tasks (like Masked Language Modeling).
# If you just mean-pool their outputs out-of-the-box, the resulting vector space is 
# often "anisotropic" (all embeddings clump together in a narrow cone), making cosine 
# similarity highly ineffective. 
# 
# SentenceTransformers (SBERT) solve this by explicitly fine-tuning the models using 
# Siamese network structures. They are specifically trained to produce high-quality, 
# semantically meaningful sentence embeddings where cosine similarity actually works.

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Frase 1: 'El gato duerme tranquilamente en el sofá.'
Frase 2: 'Un felino descansa plácidamente sobre el sillón.'
Similitud Coseno: 0.5020


## Actividad 3 — Paráfrasis

Escribe dos paráfrasis de una misma idea y muestra que sus embeddings (sentence-transformers) tienen **mayor** similitud entre sí que con una frase de tema distinto.


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Cargamos el modelo
model = SentenceTransformer('all-MiniLM-L6-v2')

# 1. Definimos las tres frases
# Dos frases que dicen lo mismo con distintas palabras (Paráfrasis)
para1 = "The car ran out of gas on the highway."
para2 = "The vehicle depleted its fuel while driving on the freeway."

# Una frase de un tema completamente distinto (Ruido)
unrelated = "Baking chocolate chip cookies makes the house smell great."

# 2. Generamos los embeddings
embeddings = model.encode([para1, para2, unrelated])

# 3. Calculamos la similitud coseno (usando la matemática de NumPy)
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim_parafra = cosine_sim(embeddings[0], embeddings[1]) # para1 vs para2
sim_distinta = cosine_sim(embeddings[0], embeddings[2]) # para1 vs unrelated

# 4. Mostramos y verificamos los resultados
print("Frase Base:      ", para1)
print("Frase Paráfrasis:", para2)
print("Frase Distinta:  ", unrelated, "\n")

print(f"Similitud Base vs Paráfrasis: {sim_parafra:.4f}")
print(f"Similitud Base vs Distinta:   {sim_distinta:.4f}\n")

# Verificamos programáticamente que la hipótesis se cumple
assert sim_parafra > sim_distinta, "Error: La frase distinta tiene más similitud."
print("¡Éxito! Las paráfrasis tienen una similitud mucho mayor.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Frase Base:       The car ran out of gas on the highway.
Frase Paráfrasis: The vehicle depleted its fuel while driving on the freeway.
Frase Distinta:   Baking chocolate chip cookies makes the house smell great. 

Similitud Base vs Paráfrasis: 0.5975
Similitud Base vs Distinta:   0.0499

¡Éxito! Las paráfrasis tienen una similitud mucho mayor.
